# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kiran162005/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### My Baseline Rule

I will create a simple content-review priority score using two observable Search Console signals: **search volume and CTR**.

A page receives a higher score when it has enough search impressions and a low observed CTR. Pages with high search volume but a higher CTR are placed in a lower-priority WATCH group. Pages below the minimum volume threshold receive NO_ACTION.

The rule is intended as a transparent baseline for prioritizing pages for human review. It is a decision-support rule and does not claim that a page definitely needs a content update or that updating it will improve performance.

### Reason Codes

* **LOW_CTR_HIGH_VOLUME** — the page has at least 100 impressions and an observed CTR below 2%, creating a possible CTR review opportunity.
* **HIGH_SEARCH_VOLUME** — the page has at least 100 impressions but its observed CTR is not below 2%, so it is placed in WATCH.
* **LOW_PRIORITY** — the page does not meet the minimum 100-impression threshold.

In [4]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN exists:", bool(HF_TOKEN))
print("HF_TOKEN starts correctly:", HF_TOKEN.startswith("hf_") if HF_TOKEN else False)

con = duckdb.connect()

con.execute("DROP SECRET IF EXISTS hf")

con.execute(f"""
CREATE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)

""")

print("DuckDB connected to Hugging Face.")

HF_TOKEN exists: True
HF_TOKEN starts correctly: True
DuckDB connected to Hugging Face.


In [6]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing from Colab Secrets.")

con = duckdb.connect()

con.execute("DROP SECRET IF EXISTS hf")

con.execute(
    f"CREATE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN.strip()}')"
)

FACT = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet'"
    ")"
)

print("Connection ready.")
print(FACT)

Connection ready.
read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')


In [7]:
test = con.sql(
    f"""
    SELECT
        COUNT(*) AS rows_in_march,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {FACT}
    """
).df()

display(test)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_in_march,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [4]:
print(march_data.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline rule

I aggregate March 2026 Search Console observations to the client-content level and rank pages using observed impressions and CTR.

A page with at least 100 impressions and CTR below 2% receives the highest score and a REVIEW action. A page with at least 100 impressions but CTR at or above 2% receives a WATCH action. Pages below the volume threshold receive NO_ACTION.

The score uses only observed March 2026 signals. No future-window performance, product-generated flags, or label-derived fields are used.

The ranked queue is written to `work/outputs/baseline_action_score.csv`.

In [8]:
baseline = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        AVG(gsc_avg_position) AS avg_position
    FROM {FACT}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
    """
).df()

print("Content-level rows:", len(baseline))
display(baseline.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Content-level rows: 176738


,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,0.001112,5.145765
1,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,0.000000,4.909314
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,329.0,0.0,0.000000,5.177774
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,772.0,1.0,0.001295,4.685335
4,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,14.0,0.0,0.000000,4.266667


In [9]:
print(FACT)

read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')


In [10]:
baseline["score"] = 0
baseline["reason_code"] = "LOW_PRIORITY"
baseline["action"] = "NO_ACTION"

high_volume = baseline["impressions"] >= 100
low_ctr = baseline["ctr"] < 0.02

baseline.loc[high_volume, "score"] = 1
baseline.loc[high_volume, "reason_code"] = "HIGH_SEARCH_VOLUME"
baseline.loc[high_volume, "action"] = "WATCH"

baseline.loc[high_volume & low_ctr, "score"] = 2
baseline.loc[high_volume & low_ctr, "reason_code"] = "LOW_CTR_HIGH_VOLUME"
baseline.loc[high_volume & low_ctr, "action"] = "REVIEW"

baseline = baseline.sort_values(
    ["score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

display(baseline.head(20))

,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,score,reason_code,action
0,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,0.009185,2.383011,2,LOW_CTR_HIGH_VOLUME,REVIEW
1,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,0.006034,2.854514,2,LOW_CTR_HIGH_VOLUME,REVIEW
2,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,0.002731,15.008339,2,LOW_CTR_HIGH_VOLUME,REVIEW
3,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,0.003253,2.675217,2,LOW_CTR_HIGH_VOLUME,REVIEW
4,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,0.000113,7.346909,2,LOW_CTR_HIGH_VOLUME,REVIEW
5,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,0.004187,3.367835,2,LOW_CTR_HIGH_VOLUME,REVIEW
6,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,205045.0,2446.0,0.011929,4.544203,2,LOW_CTR_HIGH_VOLUME,REVIEW
7,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,0.001420,2.563756,2,LOW_CTR_HIGH_VOLUME,REVIEW
8,client_62f4a7e64f5e0096,content_f107e54b10b43725,195997.0,996.0,0.005082,3.186054,2,LOW_CTR_HIGH_VOLUME,REVIEW
9,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,0.001244,32.766674,2,LOW_CTR_HIGH_VOLUME,REVIEW


In [11]:
print("Actions:")
display(baseline["action"].value_counts())

print("\nReason codes:")
display(baseline["reason_code"].value_counts())

Actions:


,count
action,
REVIEW,100660
NO_ACTION,75297
WATCH,781



Reason codes:


,count
reason_code,
LOW_CTR_HIGH_VOLUME,100660
LOW_PRIORITY,75297
HIGH_SEARCH_VOLUME,781


In [12]:
import os

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved successfully.")
print("Rows:", len(baseline))
print("Path: work/outputs/baseline_action_score.csv")

Saved successfully.
Rows: 176738
Path: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:
top20 = baseline.head(20).copy()

display(
    top20[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "clicks",
            "ctr",
            "avg_position",
            "score",
            "reason_code",
            "action"
        ]
    ]
)

,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,score,reason_code,action
0,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,0.009185,2.383011,2,LOW_CTR_HIGH_VOLUME,REVIEW
1,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,0.006034,2.854514,2,LOW_CTR_HIGH_VOLUME,REVIEW
2,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,0.002731,15.008339,2,LOW_CTR_HIGH_VOLUME,REVIEW
3,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,0.003253,2.675217,2,LOW_CTR_HIGH_VOLUME,REVIEW
4,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,0.000113,7.346909,2,LOW_CTR_HIGH_VOLUME,REVIEW
5,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,0.004187,3.367835,2,LOW_CTR_HIGH_VOLUME,REVIEW
6,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,205045.0,2446.0,0.011929,4.544203,2,LOW_CTR_HIGH_VOLUME,REVIEW
7,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,0.001420,2.563756,2,LOW_CTR_HIGH_VOLUME,REVIEW
8,client_62f4a7e64f5e0096,content_f107e54b10b43725,195997.0,996.0,0.005082,3.186054,2,LOW_CTR_HIGH_VOLUME,REVIEW
9,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,0.001244,32.766674,2,LOW_CTR_HIGH_VOLUME,REVIEW


In [14]:
top20_review = top20[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "score",
        "reason_code",
        "action"
    ]
].copy()

top20_review["confidence_note"] = (
    "Directional baseline based on observed March 2026 search signals."
)

top20_review["what_would_make_it_wrong"] = (
    "The pick could be wrong if the low CTR reflects query intent, "
    "SERP features, seasonality, or insufficient historical context."
)

display(top20_review)

,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,score,reason_code,action,confidence_note,what_would_make_it_wrong
0,client_e547b89c05043229,content_eadb33b5df496f4a,617124.0,5668.0,0.009185,2.383011,2,LOW_CTR_HIGH_VOLUME,REVIEW,Directional baseline based on observed March 2...,The pick could be wrong if the low CTR reflect...
1,client_e547b89c05043229,content_ec2e0346994fb5a5,245276.0,1480.0,0.006034,2.854514,2,LOW_CTR_HIGH_VOLUME,REVIEW,Directional baseline based on observed March 2...,The pick could be wrong if the low CTR reflect...
2,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,0.002731,15.008339,2,LOW_CTR_HIGH_VOLUME,REVIEW,Directional baseline based on observed March 2...,The pick could be wrong if the low CTR reflect...
3,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,0.003253,2.675217,2,LOW_CTR_HIGH_VOLUME,REVIEW,Directional baseline based on observed March 2...,The pick could be wrong if the low CTR reflect...
4,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,0.000113,7.346909,2,LOW_CTR_HIGH_VOLUME,REVIEW,Directional baseline based on observed March 2...,The pick could be wrong if the low CTR reflect...
5,client_62f4a7e64f5e0096,content_7172a7fad43f0998,205867.0,862.0,0.004187,3.367835,2,LOW_CTR_HIGH_VOLUME,REVIEW,Directional baseline based on observed March 2...,The pick could be wrong if the low CTR reflect...
6,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,205045.0,2446.0,0.011929,4.544203,2,LOW_CTR_HIGH_VOLUME,REVIEW,Directional baseline based on observed March 2...,The pick could be wrong if the low CTR reflect...
7,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,0.001420,2.563756,2,LOW_CTR_HIGH_VOLUME,REVIEW,Directional baseline based on observed March 2...,The pick could be wrong if the low CTR reflect...
8,client_62f4a7e64f5e0096,content_f107e54b10b43725,195997.0,996.0,0.005082,3.186054,2,LOW_CTR_HIGH_VOLUME,REVIEW,Directional baseline based on observed March 2...,The pick could be wrong if the low CTR reflect...
9,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,0.001244,32.766674,2,LOW_CTR_HIGH_VOLUME,REVIEW,Directional baseline based on observed March 2...,The pick could be wrong if the low CTR reflect...



I reviewed the top 20 rows produced by the baseline ranking. The action and reason code come directly from the rule. Confidence is directional because this is a rule-based baseline, not a validated causal model.

For each row, I considered what could make the recommendation wrong, including search intent, SERP conditions, seasonality, and limited historical context.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


### Weak picks

Some of the ranked picks are weaker than others even though they received the same `LOW_CTR_HIGH_VOLUME` reason code.

- **`content_512dbad65bd5ade9`** is a relatively weak pick because its CTR is **1.62%**, which is higher than most of the other Top-20 rows. The rule may be over-prioritizing it because of its high impression volume.
- **`content_e7b5dd4dff461ad2`** is less certain because it has a CTR of **1.19%** and an average position of **4.54**. The lower CTR may be related to query mix or SERP behavior rather than a content problem.
- **`content_36e53e9c707674fc`** is a weaker content-refresh candidate because its average position is **32.77**. Its low CTR may mainly reflect poor ranking rather than a content issue.
- **`content_82e35c4845e6c391`** has a similar limitation because its average position is **22.56**. Low CTR may be explained by ranking rather than content quality.

These examples show that **high impressions + low CTR is useful for prioritization, but it does not prove that refreshing the content will improve performance**.

### Leakage check

The baseline uses only observed March 2026 search-performance fields available at the decision point:

- `gsc_impressions`
- `gsc_clicks`
- calculated `ctr`
- `gsc_avg_position`

The baseline does **not** use:

- future-period performance
- future-window labels
- `trend_direction`
- `trend_pct`
- product-generated flags or health scores
- client names, domains, queries, or keywords

The action is therefore **decision-support**, not a claim that the page will improve after a refresh.

**Leakage verdict: PASS — no future-window or product-flag inputs were used in the baseline score.**

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.